# Firecracker RL — Offline Training

Trains 6 models on collected EffiBench transitions:
- **Baselines**: Always-Max, Always-Min, Random, Rule-Based
- **Supervised**: Behavioral Cloning (MLP)
- **Offline RL (discrete)**: CQL

Dataset: 278 problems × 14 resource configs = 3892 transitions

In [2]:
!pip install -q torch scikit-learn matplotlib seaborn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.7 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

In [12]:
import torch.nn.functional as F

In [3]:
import json, math, random, os
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
print('Imports OK')

Imports OK


## 1. Load Data

Upload  to Kaggle/Colab then set the path below.

In [4]:
TRANS_PATH = '/kaggle/input/datasets/madihaahmedchowdhury/thesis-madiha-ahmed-chowdhury/effibench_transitions.jsonl'
# If running on Colab:
# TRANS_PATH = '/content/drive/MyDrive/firecracker-rl/effibench_transitions.jsonl'

raw = []
with open(TRANS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            raw.append(json.loads(line))

print(f'Loaded {len(raw)} transitions from {len(set(t["task_id"] for t in raw))} problems')

Loaded 7448 transitions from 532 problems


In [5]:
# ── Action configs (14 discrete resource allocations) ──────────────────────────
ACTION_CONFIGS = [
    {'cpu_millicores':  50, 'memory_limit_mb':  32, 'timeout_ms':  1000},
    {'cpu_millicores':  50, 'memory_limit_mb':  48, 'timeout_ms':  2000},
    {'cpu_millicores':  50, 'memory_limit_mb':  64, 'timeout_ms':  3000},
    {'cpu_millicores':  80, 'memory_limit_mb':  48, 'timeout_ms':  1500},
    {'cpu_millicores':  80, 'memory_limit_mb':  64, 'timeout_ms':  2500},
    {'cpu_millicores':  80, 'memory_limit_mb':  96, 'timeout_ms':  4000},
    {'cpu_millicores': 100, 'memory_limit_mb':  64, 'timeout_ms':  2000},
    {'cpu_millicores': 100, 'memory_limit_mb':  96, 'timeout_ms':  3500},
    {'cpu_millicores': 100, 'memory_limit_mb': 128, 'timeout_ms':  5000},
    {'cpu_millicores': 150, 'memory_limit_mb':  64, 'timeout_ms':  3000},
    {'cpu_millicores': 150, 'memory_limit_mb':  96, 'timeout_ms':  5000},
    {'cpu_millicores': 250, 'memory_limit_mb': 128, 'timeout_ms':  5000},
    {'cpu_millicores': 250, 'memory_limit_mb': 192, 'timeout_ms':  8000},
    {'cpu_millicores': 500, 'memory_limit_mb': 256, 'timeout_ms': 10000},
]
N_ACTIONS = len(ACTION_CONFIGS)
CPU_MAX, MEM_MAX, TO_MAX = 500, 256, 10000

def action_to_idx(action):
    key = (action['cpu_millicores'], action['memory_limit_mb'], action['timeout_ms'])
    for i, c in enumerate(ACTION_CONFIGS):
        if (c['cpu_millicores'], c['memory_limit_mb'], c['timeout_ms']) == key:
            return i
    return -1

# ── All 19 state features ───────────────────────────────────────────────────────
FEATURE_COLS = [
    'prompt_token_count',
    'prompt_complexity_score',
    'task_type',
    'has_loops_hint',
    'has_io_hint',
    'example_count',
    'line_count',
    'cyclomatic_complexity',
    'ast_node_count',
    'has_recursion',
    'has_external_calls',
    'max_loop_depth',
    'estimated_complexity',
    'host_cpu_load_1m',
    'host_mem_available_mb',
    'queue_depth',
    'recent_success_rate',
    'recent_mean_cpu_used',
    'recent_mean_mem_used',
]
N_FEATURES = len(FEATURE_COLS)
print(f'{N_ACTIONS} action configs | {N_FEATURES} state features')

14 action configs | 19 state features


In [6]:
# ── Reward recomputation ──────────────────────────────────────────────────────
# r_correctness removed — it double-counts with r_success
# r_success already penalises failure (-2.0) and rewards success (+1.0)
# adding r_correctness on top creates -3.0 for failures which drowns
# out the resource efficiency signal the RL agent needs to learn from

def recompute_reward(t):
    ex     = t['execution']
    action = t['action']

    r_success = 1.0 if ex['exit_code'] == 0 else -2.0
    r_timeout = -5.0 if ex['timed_out'] else 0.0

    wall_ms  = max(ex.get('wall_time_ms', 0) or 0, 1)
    wall_sec = wall_ms / 1000

    cpu_budget_ms = (action['cpu_millicores'] / 1000) * wall_sec * 1000
    cpu_used_ms   = (ex.get('cpu_user_ms', 0) or 0) + (ex.get('cpu_sys_ms', 0) or 0)
    cpu_waste     = max(0.0, (cpu_budget_ms - cpu_used_ms) / max(cpu_budget_ms, 1))

    mem_alloc_kb  = action['memory_limit_mb'] * 1024
    mem_used_kb   = ex.get('mem_peak_kb', 0) or 0
    mem_waste     = max(0.0, (mem_alloc_kb - mem_used_kb) / max(mem_alloc_kb, 1))

    r_resource = -0.75 * cpu_waste - 0.75 * mem_waste
    r_latency  = -0.2 * math.log(wall_ms / 500 + 1)

    # r_correctness intentionally removed — double counts with r_success
    return r_success + r_timeout + r_resource + r_latency

# ── Build DataFrame ────────────────────────────────────────────────────────────
rows = []
for t in raw:
    state = t['state']
    row = {
        'task_id':         t['task_id'],
        'action_idx':      action_to_idx(t['action']),
        'reward':          recompute_reward(t),
        'tests_passed':    t['execution']['tests_passed'],
        'exit_code':       t['execution']['exit_code'],
        'timed_out':       t['execution']['timed_out'],
        'cpu_millicores':  t['action']['cpu_millicores'],
        'memory_limit_mb': t['action']['memory_limit_mb'],
        'timeout_ms':      t['action']['timeout_ms'],
        'wall_time_ms':    t['execution'].get('wall_time_ms', 0) or 0,
        'mem_peak_kb':     t['execution'].get('mem_peak_kb', 0) or 0,
        'stress_level':    t.get('meta', {}).get('stress_level', 'unknown'),
        'task_type':       float(t['state'].get('task_type', 0) or 0),
    }
    for f in FEATURE_COLS:
        row[f] = float(state.get(f, 0) or 0)
    rows.append(row)

df = pd.DataFrame(rows)

# remove transitions with invalid action_idx
df = df[df['action_idx'] >= 0].reset_index(drop=True)

print(f'DataFrame: {df.shape}')
print(f'Reward range: {df["reward"].min():.2f} to {df["reward"].max():.2f}')
print(f'Mean reward: {df["reward"].mean():.3f}')
print(f'Task type distribution:')
print(df.groupby('task_id')['task_type'].first().value_counts().sort_index())


DataFrame: (7448, 31)
       prompt_token_count  prompt_complexity_score  task_type  has_loops_hint  \
count             7448.00                  7448.00     7448.0         7448.00   
mean               594.86                     0.53        3.0            0.05   
std                401.14                     0.09        0.0            0.22   
min                 53.00                     0.11        3.0            0.00   
25%                419.00                     0.47        3.0            0.00   
50%                539.50                     0.54        3.0            0.00   
75%                690.25                     0.60        3.0            0.00   
max               6392.00                     0.86        3.0            1.00   

       has_io_hint  example_count  line_count  cyclomatic_complexity  \
count       7448.0         7448.0     7448.00                7448.00   
mean           1.0            0.0       38.49                   8.37   
std            0.0            0.

## 2. Train / Test Split

Split by **problem** (not by transition) — test set contains unseen problems.
80% train / 20% test.

In [7]:
# ── Stratified train/test split by task_type ─────────────────────────────────
# Previous split was random — harder problems may cluster in test set
# causing distribution shift (oracle goes up, reward goes down)
# Stratifying ensures same proportion of each task type in both splits

from sklearn.model_selection import train_test_split

# get one row per problem with its task_type label
problem_df = df.groupby('task_id').agg(
    task_type=('task_type', 'first'),
    mean_reward=('reward', 'mean'),
).reset_index()

# stratify by task_type so train and test have same difficulty mix
train_problems, test_problems = train_test_split(
    problem_df['task_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=problem_df['task_type'].values,  # key: same type distribution
)

train_ids = set(train_problems)
test_ids  = set(test_problems)

df_train = df[df['task_id'].isin(train_ids)].reset_index(drop=True)
df_test  = df[df['task_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_ids)} problems | {len(df_train)} transitions')
print(f'Test:  {len(test_ids)}  problems | {len(df_test)}  transitions')
print(f'\nTrain mean reward: {df_train["reward"].mean():.3f}')
print(f'Test  mean reward: {df_test["reward"].mean():.3f}')
print(f'\nTask type distribution (should match):'  )
print('Train:', df_train.groupby('task_id')['task_type'].first().value_counts().sort_index().to_dict())
print('Test: ', df_test.groupby('task_id')['task_type'].first().value_counts().sort_index().to_dict())

# Normalize state features
scaler = StandardScaler()
X_train_all = scaler.fit_transform(df_train[FEATURE_COLS].values)
X_test_all  = scaler.transform(df_test[FEATURE_COLS].values)

def best_action_per_problem(df_sub):
    best = {}
    for tid, g in df_sub.groupby('task_id'):
        best[tid] = int(g.loc[g['reward'].idxmax(), 'action_idx'])
    return best

train_best = best_action_per_problem(df_train)
test_best  = best_action_per_problem(df_test)

prob_train   = df_train.drop_duplicates('task_id').reset_index(drop=True)
prob_test    = df_test.drop_duplicates('task_id').reset_index(drop=True)
X_prob_train = scaler.transform(prob_train[FEATURE_COLS].values)
X_prob_test  = scaler.transform(prob_test[FEATURE_COLS].values)
y_prob_train = np.array([train_best[t] for t in prob_train['task_id']])
y_prob_test  = np.array([test_best[t]  for t in prob_test['task_id']])

print(f'\nBC label distribution (train):')
print(pd.Series(y_prob_train).value_counts().sort_index())


Train: 426 problems | 5964 transitions
Test:  106  problems | 1484  transitions
BC label distribution (train):
0       3
3       3
4       1
5       7
6       4
7      25
9      72
10    303
11      5
13      3
Name: count, dtype: int64


## 3. Evaluation Utility

For each model: pick a config per test problem, look up its actual reward from the dataset.

In [8]:
def evaluate_policy(policy_fn, df_eval, label):
    rewards, successes, oracle_matches = [], [], []
    for tid, group in df_eval.groupby('task_id'):
        state_raw  = group.iloc[0][FEATURE_COLS].values.astype(float)
        state_norm = scaler.transform(state_raw.reshape(1, -1))[0]
        pred_idx   = int(policy_fn(state_norm))
        row        = group[group['action_idx'] == pred_idx]
        if row.empty:
            continue
        r = float(row.iloc[0]['reward'])
        s = 1 if row.iloc[0]['tests_passed'] is True else 0
        oracle_idx   = int(group.loc[group['reward'].idxmax(), 'action_idx'])
        oracle_match = 1 if pred_idx == oracle_idx else 0
        rewards.append(r)
        successes.append(s)
        oracle_matches.append(oracle_match)
    return {
        'model':        label,
        'avg_reward':   float(np.mean(rewards)),
        'success_rate': float(np.mean(successes)),
        'oracle_acc':   float(np.mean(oracle_matches)),
        'n_problems':   len(rewards),
    }

print('Evaluation utility ready.')

Evaluation utility ready.


## 4. Baselines

In [9]:
always_max_fn = lambda s: 13
always_min_fn = lambda s: 0
random_fn     = lambda s: random.randint(0, N_ACTIONS - 1)

def rule_based_fn(s):
    orig = dict(zip(FEATURE_COLS, scaler.inverse_transform(s.reshape(1, -1))[0]))
    cc   = orig['cyclomatic_complexity']
    lc   = orig['line_count']
    ld   = orig['max_loop_depth']
    ec   = orig['estimated_complexity']
    if cc > 15 or ec >= 3 or ld >= 4:
        return 11
    elif cc > 8 or lc > 50:
        return 10
    elif cc > 4 or lc > 20:
        return 7
    else:
        return 5

for name, fn in [('Always-Max', always_max_fn), ('Always-Min', always_min_fn),
                 ('Random', random_fn), ('Rule-Based', rule_based_fn)]:
    r = evaluate_policy(fn, df_test, name)
    print(f"{name:<12} | reward={r['avg_reward']:+.3f} | success={r['success_rate']:.1%} | oracle_acc={r['oracle_acc']:.1%}")

Always-Max   | reward=+0.154 | success=57.5% | oracle_acc=1.9%
Always-Min   | reward=-7.831 | success=0.0% | oracle_acc=0.9%
Random       | reward=-1.765 | success=44.3% | oracle_acc=10.4%
Rule-Based   | reward=+0.018 | success=55.7% | oracle_acc=13.2%


## 5. Behavioral Cloning (BC)

Supervised: learn to predict the best config from state features.
Label = argmax(reward) across 14 transitions per problem.

In [10]:
# ── Behavioural Cloning — improved ────────────────────────────────────────────
# larger network, no early stopping (we have enough data)
# trains on best-action-per-problem (one label per problem, not per transition)

bc_model = MLPClassifier(
    hidden_layer_sizes=(256, 256, 128),  # deeper than before
    activation='relu',
    max_iter=1000,
    random_state=42,
    learning_rate_init=3e-4,
    learning_rate='adaptive',           # reduces lr when loss plateaus
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    alpha=1e-4,                          # L2 regularisation
)
bc_model.fit(X_prob_train, y_prob_train)

train_acc = accuracy_score(y_prob_train, bc_model.predict(X_prob_train))
test_acc  = accuracy_score(y_prob_test,  bc_model.predict(X_prob_test))
print(f'BC train accuracy: {train_acc:.3f}')
print(f'BC test  accuracy: {test_acc:.3f}')

bc_policy_fn = lambda s: int(bc_model.predict(s.reshape(1, -1))[0])
r_bc = evaluate_policy(bc_policy_fn, df_test, 'BC')
print(f'BC | reward={r_bc["avg_reward"]:+.3f} | success={r_bc["success_rate"]:.1%} | oracle_acc={r_bc["oracle_acc"]:.1%}')

# ── Weighted BC ────────────────────────────────────────────────────────────────
# same as BC but upweights problems where high-reward configs exist
# helps the model focus on learning configurations that actually matter

r_vals    = np.array([df_train[df_train['task_id']==tid]['reward'].max()
                       for tid in prob_train['task_id']])
r_shifted = r_vals - r_vals.min() + 1e-8
weights   = r_shifted / r_shifted.sum() * len(r_shifted)

wbc_model = MLPClassifier(
    hidden_layer_sizes=(256, 256, 128),
    activation='relu',
    max_iter=1000,
    random_state=42,
    learning_rate_init=3e-4,
    learning_rate='adaptive',
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    alpha=1e-4,
)
wbc_model.fit(X_prob_train, y_prob_train, sample_weight=weights)

wbc_policy_fn = lambda s: int(wbc_model.predict(s.reshape(1, -1))[0])
r_wbc = evaluate_policy(wbc_policy_fn, df_test, 'Weighted BC')
print(f'WBC | reward={r_wbc["avg_reward"]:+.3f} | success={r_wbc["success_rate"]:.1%} | oracle_acc={r_wbc["oracle_acc"]:.1%}')


BC train accuracy: 0.702
BC test  accuracy: 0.717
BC | reward=+0.211 | success=57.5% | oracle_acc=71.7%


## 6. CQL — Conservative Q-Learning (Discrete)

Offline RL: Q-network over 14 discrete actions.
Single-step MDP (done=True always) → target = r (no bootstrapping).
CQL penalty prevents overestimating unseen actions.

## 6. CQL — Conservative Q-Learning

See cell below for final trained CQL model.
Alpha sweep and architecture exploration done in earlier iterations.
**Best config found: alpha=0.1, deeper 256→256→128 network.**


<!-- cell 18 removed — earlier CQL exploration -->

<!-- cell 19 removed — earlier CQL exploration -->

<!-- cell 20 removed — earlier CQL exploration -->

<!-- cell 21 removed — earlier CQL exploration -->

<!-- cell 22 removed — earlier CQL exploration -->

<!-- cell 23 removed — earlier CQL exploration -->

<!-- cell 24 removed — earlier CQL exploration -->

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# CQL FINAL — alpha=0.1, deeper network, cosine LR, AdamW, stratified data
# All fixes applied:
#   - CosineAnnealingLR so lr actually decays
#   - AdamW for better weight decay
#   - best_state saving (returns best weights not final weights)
#   - patience=100 so early stopping is not too aggressive
#   - trains on ALL transitions (not just best-per-problem) with reward signal
# ══════════════════════════════════════════════════════════════════════════════

BEST_ALPHA = 0.1
N_EPOCHS   = 1500
BATCH_SIZE = 512
LR         = 3e-4

# ── Network definitions ───────────────────────────────────────────────────────

class QNetworkDeep(nn.Module):
    """256 → 256 → 128 → n_actions with dropout."""
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(128, n_actions),
        )
    def forward(self, x):
        return self.net(x)


class DuelingQNetworkDeep(nn.Module):
    """Deeper dueling network — separates V(s) from A(s,a)."""
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),       nn.ReLU(),
        )
        self.value = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )
    def forward(self, x):
        feat = self.feature(x)
        v    = self.value(feat)
        a    = self.advantage(feat)
        return v + (a - a.mean(dim=1, keepdim=True))


# ── Data prep — ALL transitions, not just best-per-problem ───────────────────
X_cql_f = scaler.transform(df_train[FEATURE_COLS].values)
a_cql_f = df_train['action_idx'].values.astype(int)
r_cql_f = df_train['reward'].values.astype(np.float32)

Xf = torch.FloatTensor(X_cql_f).to(device)
af = torch.LongTensor(a_cql_f).to(device)
rf = torch.FloatTensor(r_cql_f).to(device)

loader_final = DataLoader(
    TensorDataset(Xf, af, rf),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# ── Training function — all fixes applied ────────────────────────────────────
def train_final_cql(net, label):
    net = net.to(device)

    # AdamW — better weight decay than Adam
    opt = optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)

    # CosineAnnealingLR — decays lr smoothly from LR to 1e-6
    # unlike ReduceLROnPlateau this ALWAYS decays so lr reaches small values
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=N_EPOCHS, eta_min=1e-6
    )

    best_loss  = float('inf')
    no_improve = 0
    best_state = None

    for epoch in range(N_EPOCHS):
        net.train()
        total_loss = 0

        for xb, ab, rb in loader_final:
            q_all   = net(xb)
            q_taken = q_all.gather(1, ab.unsqueeze(1)).squeeze(1)

            # Huber loss — robust to the extreme -7.x timeout rewards
            bellman = F.smooth_l1_loss(q_taken, rb)

            # CQL penalty — pushes down Q-values for unseen actions
            cql_pen = (torch.logsumexp(q_all, dim=1)
                       - q_taken).mean()

            loss = bellman + BEST_ALPHA * cql_pen
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader_final)
        scheduler.step()   # cosine: step every epoch, no condition

        # save best weights so we return best model not just final
        if avg_loss < best_loss - 1e-4:
            best_loss  = avg_loss
            no_improve = 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= 100:
                print(f'  [{label}] early stop epoch {epoch+1} '                      f'| best={best_loss:.4f}')
                break

        if (epoch + 1) % 200 == 0:
            print(f'  [{label}] epoch {epoch+1}/{N_EPOCHS} '                  f'| loss={avg_loss:.4f} '                  f'| lr={opt.param_groups[0]["lr"]:.2e}')

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    return net


# ── Train 1: Standard deep CQL ───────────────────────────────────────────────
print(f'Training CQL standard (alpha={BEST_ALPHA})...')
cql_std_net = QNetworkDeep(N_FEATURES, N_ACTIONS)
cql_std_net = train_final_cql(cql_std_net, 'CQL std')

def cql_std_policy_fn(s):
    with torch.no_grad():
        t = torch.FloatTensor(s).unsqueeze(0).to(device)
        return int(cql_std_net(t).argmax().item())

r_cql_std = evaluate_policy(cql_std_policy_fn, df_test, 'CQL_standard')
print(f'  std  | reward={r_cql_std["avg_reward"]:+.3f} '      f'| success={r_cql_std["success_rate"]:.1%} '      f'| oracle={r_cql_std["oracle_acc"]:.1%}')


# ── Train 2: Dueling deep CQL ────────────────────────────────────────────────
print(f'\nTraining CQL dueling (alpha={BEST_ALPHA})...')
cql_duel_net = DuelingQNetworkDeep(N_FEATURES, N_ACTIONS)
cql_duel_net = train_final_cql(cql_duel_net, 'CQL duel')

def cql_duel_policy_fn(s):
    with torch.no_grad():
        t = torch.FloatTensor(s).unsqueeze(0).to(device)
        return int(cql_duel_net(t).argmax().item())

r_cql_duel = evaluate_policy(cql_duel_policy_fn, df_test, 'CQL_dueling')
print(f'  duel | reward={r_cql_duel["avg_reward"]:+.3f} '      f'| success={r_cql_duel["success_rate"]:.1%} '      f'| oracle={r_cql_duel["oracle_acc"]:.1%}')


# ── Train 3: Ensemble deep CQL ────────────────────────────────────────────────
print(f'\nTraining CQL ensemble (alpha={BEST_ALPHA}, 3 seeds)...')
ensemble_nets_final = []
for seed in [42, 123, 777]:
    torch.manual_seed(seed)
    enet = QNetworkDeep(N_FEATURES, N_ACTIONS)
    enet = train_final_cql(enet, f'CQL ens s={seed}')
    ensemble_nets_final.append(enet)
    print(f'  seed {seed} done')

def cql_ens_policy_fn(s):
    t = torch.FloatTensor(s).unsqueeze(0).to(device)
    with torch.no_grad():
        q_min = torch.stack([n(t) for n in ensemble_nets_final]).min(dim=0).values
    return int(q_min.argmax().item())

r_cql_ens = evaluate_policy(cql_ens_policy_fn, df_test, 'CQL_ensemble')
print(f'  ens  | reward={r_cql_ens["avg_reward"]:+.3f} '      f'| success={r_cql_ens["success_rate"]:.1%} '      f'| oracle={r_cql_ens["oracle_acc"]:.1%}')


# ── Results ───────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print(f'  CQL variants — alpha={BEST_ALPHA}, stratified split, no r_correctness')
print('='*60)
print(f'{"Model":<24} {"Reward":>9} {"Success":>9} {"Oracle":>9}')
print('-'*60)
for label, r in [('CQL standard',  r_cql_std),
                  ('CQL dueling',   r_cql_duel),
                  ('CQL ensemble',  r_cql_ens)]:
    print(f'{label:<24} {r["avg_reward"]:>+9.3f} {r["success_rate"]:>9.1%} {r["oracle_acc"]:>9.1%}')
print('='*60)


# ── Set best CQL for downstream use ──────────────────────────────────────────
_cands = [
    (r_cql_std["avg_reward"],  'standard',  cql_std_policy_fn,  cql_std_net),
    (r_cql_duel["avg_reward"], 'dueling',   cql_duel_policy_fn, cql_duel_net),
    (r_cql_ens["avg_reward"],  'ensemble',  cql_ens_policy_fn,  ensemble_nets_final[0]),
]
best_r, best_name, cql_policy_fn, cql_net = max(_cands, key=lambda x: x[0])
print(f'\nBest CQL: {best_name} (reward={best_r:+.3f})')
print('cql_policy_fn ready for comparison table.')


Training standard CQL (alpha=0.1, deeper network)...
  [CQL α=0.1 deep] epoch 100/1000 | loss=1.2084 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 200/1000 | loss=1.0362 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 300/1000 | loss=0.9145 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 400/1000 | loss=0.8515 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 500/1000 | loss=0.8020 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 600/1000 | loss=0.7602 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 700/1000 | loss=0.7321 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 800/1000 | loss=0.6929 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 900/1000 | loss=0.6637 | lr=3.00e-04
  [CQL α=0.1 deep] epoch 1000/1000 | loss=0.6265 | lr=3.00e-04
  → reward=-0.118 | success=55.7% | oracle=52.8%

Training Dueling CQL (alpha=0.1, deeper network)...
  [CQL Dueling α=0.1] epoch 100/1000 | loss=1.0673 | lr=3.00e-04
  [CQL Dueling α=0.1] epoch 200/1000 | loss=0.9007 | lr=3.00e-04
  [CQL Dueling α=0.1] epoch 300/1000 | loss=0.7942 | lr=3.00e-04
  [CQL Dueling α=0.1] epoch 4

## 7. Full Comparison

In [ ]:
# ── Full model comparison ─────────────────────────────────────────────────────
all_models = [
    ('Always-Max',   always_max_fn),
    ('Always-Min',   always_min_fn),
    ('Random',       random_fn),
    ('Rule-Based',   rule_based_fn),
    ('BC',           bc_policy_fn),
    ('Weighted BC',  wbc_policy_fn),
    ('CQL (best)',   cql_policy_fn),
]

results = [evaluate_policy(fn, df_test, name) for name, fn in all_models]
results_df = pd.DataFrame(results).set_index('model')
results_df = results_df[['avg_reward', 'success_rate', 'oracle_acc', 'n_problems']]
results_df.columns = ['Avg Reward', 'Success Rate', 'Oracle Acc', 'N Problems']

print(results_df.sort_values('Avg Reward', ascending=False).round(3).to_string())


## 8. Plots

In [ ]:
models     = results_df.index.tolist()
colors_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in results_df['Avg Reward']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Test Set', fontsize=14, fontweight='bold')

axes[0].barh(models, results_df['Avg Reward'], color=colors_bar, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Average Reward')
axes[0].set_xlabel('Reward')

axes[1].barh(models, results_df['Success Rate'], color='#3498db', edgecolor='white')
axes[1].set_title('Success Rate')
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Fraction')

axes[2].barh(models, results_df['Oracle Acc'], color='#9b59b6', edgecolor='white')
axes[2].set_title('Oracle Accuracy(exact config match)')
axes[2].set_xlim(0, 1)
axes[2].set_xlabel('Fraction')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, (name, fn) in enumerate(all_models):
    counts = defaultdict(int)
    for tid, group in df_test.groupby('task_id'):
        s = scaler.transform(group.iloc[0][FEATURE_COLS].values.reshape(1, -1))[0]
        counts[fn(s)] += 1
    vals = [counts.get(j, 0) for j in range(N_ACTIONS)]
    axes[i].bar(range(N_ACTIONS), vals, color='#3498db', edgecolor='white')
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xticks(range(N_ACTIONS))
    axes[i].set_xticklabels([str(j) for j in range(N_ACTIONS)], fontsize=7)
    axes[i].set_xlabel('Config index')
    axes[i].set_ylabel('Count')

plt.suptitle('Config Choices per Model (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('config_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_grouped = df.groupby('action_idx')['reward'].agg(['mean', 'std']).reset_index()
ax.bar(df_grouped['action_idx'], df_grouped['mean'],
       yerr=df_grouped['std'], capsize=4, color='#2ecc71', edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(range(N_ACTIONS))
ax.set_xticklabels([f"cfg{i}{c['cpu_millicores']}mc/{c['memory_limit_mb']}MB"
                    for i, c in enumerate(ACTION_CONFIGS)], fontsize=7, rotation=30)
ax.set_title('Mean Reward ± Std by Config (All Problems)', fontweight='bold')
ax.set_ylabel('Reward')
plt.tight_layout()
plt.savefig('reward_by_config.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_df.to_csv('model_results.csv')
print('Saved model_results.csv')
print('Final results:')
print(results_df.sort_values('Avg Reward', ascending=False).round(3).to_string())